In [37]:
import pandas as pd
import os
import numpy as np
"""
For each grouping sheet(H-B)
"""

## grouping
# sheet_name = 'H2O - FON'
# sheet_name = 'OH - FON'
# sheet_name = 'NH - FON'
# sheet_name = 'COOH Organic Acid - FON'
# sheet_name = 'H2O - OH'
# sheet_name = 'OH - OH'
# sheet_name = 'NH - OH'
# sheet_name = 'NH - H2O'
# sheet_name = 'COOH - COOH'
# sheet_name = 'COOH - OH'
# sheet_name = 'COOH - H2O'
## sheet_name = 'COOH - NH'


## grouping-I
# sheet_name = 'Non-HB_Non-HB'
# sheet_name = 'Non-HB_HB-A'
# sheet_name = 'HB-A_HB-A'

## grouping-II
# sheet_name = 'Non-HB_H2O'
# sheet_name = 'Non-HB_OH'
# sheet_name = 'Non-HB_NH'
# sheet_name = 'Non-HB_COOH'

## group_alldata
sheet_name = 'grouping_alldata'

# grouping = pd.read_excel(r"grouping-F.xlsx", sheet_name=sheet_name)
# grouping = pd.read_excel(r"grouping-I-F.xlsx", sheet_name=sheet_name)
# grouping = pd.read_excel(r"grouping-II-F.xlsx", sheet_name=sheet_name)
# grouping = pd.read_excel(r"D:\COSMOSAC-master\COSMO\COSMO_all\ML\VLE\Phase diagram\grouping\grouping_alldata.xlsx", sheet_name=sheet_name)


# train_exp_final = pd.read_csv(r"train_exp_datapoints_final.csv")

# merged_data = pd.merge(grouping, train_exp_final, on=['comp.A', 'comp.B', 'T'], how='inner')
# xA_values = merged_data[['comp.A','comp.B','T','xA','yA_exp','P_exp']]
# xA_values.insert(xA_values.columns.get_loc('yA_exp') + 1, 'yA_pred', '')  
# xA_values.insert(xA_values.columns.get_loc('P_exp') + 1, 'P_pred', '')   

# print(xA_values)

# cleaned_sheet_name = sheet_name.replace("-", "_") # change the file name
# csv_file_path = os.path.join('VLE_grouping_file', f"{sheet_name}.csv")
# csv_file_path = os.path.join('VLE_grouping_I_file', f"{sheet_name}.csv")
# csv_file_path = os.path.join('VLE_grouping_II_file', f"{sheet_name}.csv")
# csv_file_path = os.path.join('D:\COSMOSAC-master\COSMO\COSMO_all\ML\VLE\Phase diagram\grouping', f"{sheet_name}.csv")

# xA_values.to_csv(csv_file_path, index=False)
# print(f"File saved as: {csv_file_path}")

In [38]:
import time
import csv
import pandas as pd
import numpy as np
import pickle as pk
import os
from sklearn.utils import shuffle
import tensorflow as tf
from collections import namedtuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from scipy.interpolate import InterpolatedUnivariateSpline 
import matplotlib.font_manager as fm
from optuna.samplers import TPESampler
from joblib import dump, load
# total_start_time = time.time()
from tensorflow.keras.models import load_model

In [39]:
data = pd.read_csv(r'C:\Users\Hung\Desktop\2025_D-MPNN\grouping\data\train_exp_data_DMPNN.csv', header=None)

data.iloc[:, -1] = data.iloc[:, -1]/(10**3)
# print(data)

"""
AAD-y = 1.04%  AARD-p = 2.88% 
"""

fold_idx = 8
model = load_model(rf'C:\Users\Hung\Desktop\2025_D-MPNN\model_train\results_V1\model\h5\MLP_Vali_DMPNN_{fold_idx}.h5')
pipeline = load(r"C:\Users\Hung\Desktop\2025_D-MPNN\model_train\results_V1\model\pipelineMLP_DMPNN.joblib")

scaler = pipeline.named_steps['scaler']
scaler_maxy=scaler.data_max_[-2]
scaler_maxp=scaler.data_max_[-1]
scaler_miny=scaler.data_min_[-2]
scaler_minp=scaler.data_min_[-1]

c:\Users\Hung\anaconda3\envs\tf_CPU\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.2.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Hung\anaconda3\envs\tf_CPU\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.2.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [40]:
# data = pd.read_csv(r'C:\Users\Hung\Desktop\2025_ECFP\grouping\data\train_exp_data_ECFP6_V1.csv', header=None)

# # data.iloc[:, -1] = data.iloc[:, -1]/(10**3)
# # # print(data)

# """
# AAD-y = 1.04%  AARD-p = 2.88% 
# """

# fold_idx = 1
# model = load_model(rf'C:\Users\Hung\Desktop\2025_ECFP\model_train\results_V1\model\h5\MLP_Vali_ecpf_{fold_idx}.h5')
# pipeline = load(r"C:\Users\Hung\Desktop\2025_ECFP\model_train\results_V1\model\pipelineMLP_ECPF6.joblib")

# scaler = pipeline.named_steps['scaler']
# scaler_maxy=scaler.data_max_[-2]
# scaler_maxp=scaler.data_max_[-1]
# scaler_miny=scaler.data_min_[-2]
# scaler_minp=scaler.data_min_[-1]

In [41]:
COSMOIsoline = namedtuple('COSMOIsoline', ['T', 'p', 'XA', 'yA'])

def get_isotherm_ml(model, T):
    
    TT, PP, X, y = [], [], [], []

    for x0L in np.linspace(1e-6, 1-1e-6):
        X = np.array([[T, X]])
        p = model.predict(X)[0]
        XA = yA  
        TT.append(T); PP.append(p); X.append(XA); y.append(yA)
    return COSMOIsoline(TT, np.array(PP), np.array(X), y)

In [42]:
# folder_path = r's-profile-area'

# def training_data(number):
#     extracted_data = []
#     file_path = folder_path + '/' + number +'-opt-b3lyp.csv'
  
#     with open(file_path, 'r', newline='', encoding='utf-8') as csv_file:
#         csv_reader = csv.reader(csv_file)
#         row_data = []
#         for row in csv_reader:
#             row_data.append(row[1:52])
#             if len(row_data) == 51:
#                 break

#         float_data = [item[0] for item in row_data]
#         extracted_data.append(float_data)
#         return extracted_data


# file_path= r'D:\COSMOSAC-master\COSMO\COSMO_all\ML\VLE\COSMO_NUMBER_CID_all.csv'

# def name_to_cosmo(COSMO_num):

#     with open(file_path, 'r', newline='', encoding='utf-8') as csv_file:
#         csv_reader = csv.reader(csv_file)
#         row_data = []
#         for row in csv_reader:
#             row_data.append(row)

#     for i in range(0,1959):
#         if (row_data[i][0]==COSMO_num):
#             return (row_data[i][0])
#     return 0

# # print(name_to_cosmo("10014"))
# # print(training_data(str(name_to_cosmo("10014"))))    

# """
# for training datasets
# """
# def train_datasets(fluids,T, x):
#     save_compounds = []
#     save_compounds.extend(training_data(str(name_to_cosmo(str(fluids[0]))))[0])
#     save_compounds.extend(training_data(str(name_to_cosmo(str(fluids[1]))))[0])
#     save_compounds.extend([str(T)])
#     save_compounds.extend([str(x)])
#     save_compounds.extend([0])
#     save_compounds.extend([0])
#     return save_compounds

In [43]:
"""
D-MPNN
"""

folder_path = r'C:\Users\Hung\Desktop\2025_D-MPNN\grouping\data\D-MPNN_all'

def training_data(number):
    extracted_data = []
    file_path = folder_path + '/' + number +'.csv'
  
    with open(file_path, 'r', newline='', encoding='utf-8') as csv_file:
        csv_reader = csv.reader(csv_file)
        row_data = []
        for row in csv_reader:
            row_data.append(row)
            # row_data.append(row[0:167]) # for MACCS
            # if len(row_data) == 167:
            #     break

        float_data = [item[0] for item in row_data]
        extracted_data.append(float_data)
        return extracted_data


file_path= r'C:\Users\Hung\Desktop\2025_D-MPNN\grouping\data\COSMO_CID_SMILES.csv'

def name_to_cosmo(COSMO_num):

    with open(file_path, 'r', newline='', encoding='utf-8') as csv_file:
        csv_reader = csv.reader(csv_file)
        row_data = []
        for row in csv_reader:
            row_data.append(row)

    for i in range(0,len(row_data)):
        if (row_data[i][0]==COSMO_num):
            return (row_data[i][0])
    return COSMO_num

# print(name_to_cosmo("10014"))
# print(training_data(str(name_to_cosmo("10014"))))    

"""
for training datasets
"""
def train_datasets(fluids,T, x):
    save_compounds = []
    save_compounds.extend(training_data(str(name_to_cosmo(str(fluids[0]))))[0])
    save_compounds.extend(training_data(str(name_to_cosmo(str(fluids[1]))))[0])
    save_compounds.extend([str(T)])
    save_compounds.extend([str(x)])
    save_compounds.extend([0])
    save_compounds.extend([0])
    return save_compounds

In [44]:
# grouping_sheet = pd.read_csv(f'VLE_grouping_file\{sheet_name}.csv')
# grouping_sheet = pd.read_csv(f'VLE_grouping_I_file\{sheet_name}.csv')
# grouping_sheet = pd.read_csv(f'VLE_grouping_II_file\{sheet_name}.csv')
grouping_sheet = pd.read_csv(rf'C:\Users\Hung\Desktop\2025_D-MPNN\grouping\data\{sheet_name}.csv')

# csv_file_path = f'VLE_grouping_finalACC_COSMO\{sheet_name}.csv'
# csv_file_path = f'VLE_grouping_I_finalACC_COSMO\{sheet_name}.csv'
# csv_file_path = f'VLE_grouping_II_finalACC_COSMO\{sheet_name}.csv'
csv_file_path = rf'C:\Users\Hung\Desktop\2025_D-MPNN\grouping\grouping_dmpnn\{sheet_name}_ACC_DMPNN.csv'


file_exists = False
try:
    with open(csv_file_path, 'r') as f:
        file_exists = True
except FileNotFoundError:
    pass  

for i in range(len(grouping_sheet)):
    compA = int(grouping_sheet['comp.A'].iloc[i])
    compB = int(grouping_sheet['comp.B'].iloc[i])
    T_values = str(grouping_sheet['T'].iloc[i])
    xA = float(grouping_sheet['xA'].iloc[i])
    yA_exp = float(grouping_sheet['yA_exp'].iloc[i])
    P_exp = float(grouping_sheet['P_exp'].iloc[i])
    # print(f"{compA}, {compB}, {T_values}, {xA}, {yA_exp}, {P_exp}")

    results = []

    for T in [T_values]:
        train_input = []
        for xA in [xA]:
            train_input.append(train_datasets([compA,compB],T, xA))
        train_input = np.array(train_input)
        # print(f"Shape of train_input for T={T}: {train_input.shape}")

        # train_input = pipeline.fit_transform(train_input)
        train_input = pipeline.named_steps['scaler'].transform(train_input)
        train_input = pd.DataFrame(train_input)
        train_input = train_input.to_numpy()
        train_input = train_input[:, 0:-2]
        # print(f"Shape of train_input for T={T}: {train_input.shape}")
        
        start_time = time.time()

        pred_output = model.predict(train_input)
        
        prediction_time = time.time() - start_time 
        
        # Normalization and skewness correction
        pred_output[:,0] = pred_output[:,0]*(scaler_maxy-scaler_miny)+scaler_miny
        pred_output[:,1] = pred_output[:,1]*(scaler_maxp-scaler_minp)+scaler_minp
        pred_output[:,1] = pred_output[:,1]**15*1000

        # x_pred = np.linspace(0, 1, num = num)
        # y_pred = np.ones(num, dtype = float)-pred_output[:,0]
        y_pred = pred_output[:,0]
        p_pred = pred_output[:,1]/10**3
        P_exp = P_exp/1000


        # total_end_time = time.time()
        # total_prediction_time = total_end_time - total_start_time

        """
        Acc
        """
        # AARD-P
        AARD_P = 100 * (np.abs((P_exp - p_pred) / P_exp))
        print(AARD_P)

        # AAD-y
        AAD_y = 100 * (np.abs((yA_exp - y_pred)))
        print(AAD_y)
 

        result = {
            'comp.A': compA,
            'comp.B': compB,
            'T(K)': T,
            'x': xA,
            'yexp': yA_exp,
            'ycal': y_pred[0],
            'Pexp(kPa)': P_exp,
            'Pcal(kPa)': p_pred[0],
            'AARD-P(%)': AARD_P[0],
            'AAD-y1(%)': AAD_y[0],
            'Pred Time (s)': float(prediction_time)
        }

        results.append(result)

        df_result = pd.DataFrame([result])  
        df_result.to_csv(csv_file_path, mode='a', header=not file_exists, index=False, quoting=csv.QUOTE_NONE)
        
        file_exists = True
print(f"File saved as: {csv_file_path}")

1/1 [==============================] - 0s 68ms/step
[nan]
[nan]
1/1 [==============================] - 0s 29ms/step
[nan]
[nan]
1/1 [==============================] - 0s 25ms/step
[nan]
[nan]
1/1 [==============================] - 0s 27ms/step
[nan]
[nan]
1/1 [==============================] - 0s 29ms/step
[nan]
[nan]
1/1 [==============================] - 0s 25ms/step
[nan]
[nan]
1/1 [==============================] - 0s 24ms/step
[nan]
[nan]
1/1 [==============================] - 0s 26ms/step
[nan]
[nan]
1/1 [==============================] - 0s 32ms/step
[nan]
[nan]
1/1 [==============================] - 0s 29ms/step
[nan]
[nan]
1/1 [==============================] - 0s 26ms/step
[nan]
[nan]
1/1 [==============================] - 0s 30ms/step
[nan]
[nan]
1/1 [==============================] - 0s 27ms/step
[nan]
[nan]
1/1 [==============================] - 0s 28ms/step
[nan]
[nan]
1/1 [==============================] - 0s 27ms/step
[nan]
[nan]
1/1 [==============================] - 0

In [45]:
# summary_data = []
# sheet = pd.read_csv(f'VLE_grouping_I_finalACC_COSMO\{sheet_name}.csv')

# # print(sheet)

# AARD_P = sheet['AARD-P(%)']
# AAD_y = sheet['AAD-y1(%)']
# PredTime = sheet['Pred Time (s)']

# Aver_AARD_P = np.mean(AARD_P)
# median_AARD_P = np.median(AARD_P) 
# Aver_AAD_y = np.mean(AAD_y)
# median_AAD_y = np.median(AAD_y)
# Aver_Pred_Time = np.mean(PredTime)

# summary = {
#     'sheet_name': sheet_name,
#     'data': len(sheet),
#     'Aver.AARD_P(%)': Aver_AARD_P,
#     'Median.AARD_P(%)': median_AARD_P,  
#     'Aver.AAD_y(%)': Aver_AAD_y,
#     'Median.AAD_y(%)': median_AAD_y,    
#     'Aver.Pred.Time(s)': Aver_Pred_Time
# }
# summary_data.append(summary)
# summary_df = pd.DataFrame(summary_data)
# # summary_df = summary_df.drop_duplicates()
# summary_csv_path = 'summary_COSMO.csv'
# summary_df.to_csv(summary_csv_path, mode='a', index=False, header=not pd.io.common.file_exists(summary_csv_path))
# print(f"統整結果已保存為：{summary_csv_path}")